# CSCIE89 HW05 – Problem 3: Transfer learning with VGG16 on the flower dataset

This notebook starts from **3_FineTune_VGG16_PyTorch_Cat_Dog** and keeps its code and structure. The changes for Problem 3 are:

- **Flower dataset instead of cats and dogs:** the Google Drive + local Colab setup from Problem 2, and the same stratified 70% / 15% / 15% split (2,569 / 550 / 551 images).
- **Five classes instead of two:** the final layer has 5 outputs, `BCEWithLogitsLoss` becomes `CrossEntropyLoss`, and predictions use `argmax` instead of `logits >= 0`.
- **Precision of the stage 1 and stage 2 models on the entire test set** (section 11).
- **One test image of every flower class** (section 12).

Steps (same as the class notebook):

1. Load and prepare the images.
2. Load pretrained VGG16 and add a new classifier.
3. Stage 1: freeze the convolutional layers and train only the classifier.
4. Stage 2: unfreeze the final convolutional block and fine-tune it with a smaller learning rate.
5. Evaluate on unused test images and save the model.

The first run downloads the flower dataset and roughly 528 MB of VGG16 weights. Use a GPU in Colab (**Runtime → Change runtime type → GPU**); training on a CPU is slow.

## 1. Import tools and choose training settings

PyTorch handles training; torchvision provides VGG16 and image tools. Scikit-learn helps split the data and report results.

- An **epoch** is one pass through all training images.
- A **batch** is a small group of images processed together.
- The **learning rate** controls the size of weight updates.
- The **seed** makes the data split repeatable; exact training results can still vary.
- The code selects an NVIDIA GPU, an Apple GPU, or a CPU, in that order.

`precision_score` is added to compute overall precision for Problem 3.

In [ ]:
# Run once if packages are missing, then restart the kernel:
# %pip install torch torchvision scikit-learn matplotlib pillow

from pathlib import Path                         # Path builds file paths that work on any OS.
import copy                                      # deepcopy keeps a frozen copy of the best weights.
import json                                      # Save metrics as a human-readable JSON file.
import torch                                     # PyTorch: tensors and training tools.
from torch import nn                             # Neural-network layers (Linear, ReLU, Dropout, ...).
from torch.utils.data import DataLoader, Subset  # DataLoader makes batches; Subset picks images by index.
from torchvision import datasets, models, transforms  # ImageFolder, pretrained VGG16, image transformations.
from torchvision.datasets.utils import download_and_extract_archive  # Download and unpack the dataset archive.
from sklearn.model_selection import train_test_split  # Reproducible, stratified data split.
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, precision_score  # Report, confusion matrix, precision.
import matplotlib.pyplot as plt                  # Plotting library.
from PIL import Image                            # Pillow opens image files.

SEED = 42                                        # Same seed as Problem 2, so the data split is the same.
IMAGE_SIZE = 150                                 # Every image is resized to 150 x 150 pixels.
BATCH_SIZE = 20                                  # Images per training step.
INITIAL_EPOCHS = 5                               # Stage 1 epochs (frozen VGG16 base).
FINETUNE_EPOCHS = 3                              # Stage 2 epochs (fine-tuning block 5).
INITIAL_LR = 2e-5                                # Stage 1 learning rate.
FINETUNE_LR = 1e-5                               # Smaller learning rate for fine-tuning.
PATIENCE = 3                                     # Early stopping: epochs to wait without improvement.

torch.manual_seed(SEED)                          # Seed PyTorch (new classifier weights, shuffling, augmentation).
torch.set_num_threads(4)                         # Avoid excessive CPU thread overhead.
device = torch.device('cuda' if torch.cuda.is_available() else          # Use an NVIDIA GPU if present,
                      'mps' if torch.backends.mps.is_available() else 'cpu')  # else Apple GPU, else the CPU.
print('Using device:', device)                   # Show the device in use.

## 2. Google Drive + local Colab setup: load the flower images

This is the same setup as Problem 2:

- A **permanent copy** of the photos is kept in Google Drive (`DRIVE_DATA_DIR`).
- Training reads the photos from Colab's **fast local disk** (`DATA_DIR = /content/data`).
- The first time, the dataset is downloaded into Colab and copied to Drive. Later sessions copy it from Drive instead of downloading it again.

The [Flower Photos dataset](https://www.tensorflow.org/tutorials/images/classification) has 3,670 photographs in five folders: daisy, dandelion, roses, sunflowers, and tulips. VGG16 weights are cached in `DATA_DIR / 'torch_hub'` and outputs are written to `outputs/vgg16_flowers`.

In [ ]:
# GOOGLE DRIVE + LOCAL COLAB SETUP

from google.colab import drive                   # Colab tool to connect your Google Drive.
import shutil                                    # Copy whole folders.

drive.mount('/content/drive')                    # Mount Google Drive (asks for permission the first time).

# Permanent location in Google Drive
PROJECT_DIR = Path('/content/drive/MyDrive/SCI-E-89-1-Deep-Learning/HW05')  # Project folder inside your Drive.
DRIVE_DATA_DIR = PROJECT_DIR / 'data'                                       # Permanent data folder in Drive.
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)                           # Create it; no error if it already exists.

# IMPORTANT: # DATA_DIR is the fast temporary Colab drive

DATA_DIR = Path('/content/data')                                            # Fast local folder; erased when the Colab session ends.
DATA_DIR.mkdir(parents=True, exist_ok=True)                                 # Create it; no error if it already exists.


flower_root = DATA_DIR / 'flower_photos'                                    # Flowers on the fast local disk (used for training).
drive_flower_root = DRIVE_DATA_DIR / 'flower_photos'                        # Permanent copy of the flowers in Drive.

# If flowers are not already in Colab
if not flower_root.exists():

    # Copy from Google Drive if already saved there
    if drive_flower_root.exists():
        shutil.copytree(drive_flower_root, flower_root)  # Drive -> local disk.

    # Otherwise download them into Colab
    else:
        download_and_extract_archive(            # Download the archive and unpack it:
            'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',  # dataset URL,
            download_root=str(DATA_DIR),         # into the local folder.
        )

# Save a permanent copy to Google Drive if one does not exist
if not drive_flower_root.exists():
    shutil.copytree(flower_root, drive_flower_root)  # Local disk -> Drive (only once).

output_dir = Path('outputs/vgg16_flowers')       # Where the model, metrics, and plot are saved.
output_dir.mkdir(parents=True, exist_ok=True)    # Create it; no error if it already exists.
torch.hub.set_dir(str(DATA_DIR / 'torch_hub'))   # Cache the downloaded VGG16 weights here.
print('Flower directory:', flower_root)          # Confirm where the photos are.

## 3. Prepare images for VGG16

Every image is resized to 150 × 150 pixels so images can be grouped into batches. `ToTensor()` converts an RGB image to three numeric channels with values from 0 to 1. `Normalize()` then adjusts each channel using the ImageNet mean and standard deviation expected by the pretrained weights.

Training images also receive random flips and small rotations. This data augmentation gives the model varied examples. Validation and test images are processed consistently, without random changes.

`ImageFolder` reads images from class-named folders and assigns their numeric labels. The flowers have a single folder, so we create **two views** of it: one with the training transform and one with the evaluation transform. Section 4 decides which images each view supplies.

In [ ]:
MEAN = [0.485, 0.456, 0.406]                     # ImageNet mean per RGB channel.
STD = [0.229, 0.224, 0.225]                      # ImageNet standard deviation per RGB channel.

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # Resize to 150 x 150.
    transforms.RandomHorizontalFlip(),     # Flip some training images left-to-right.
    transforms.RandomRotation(10),         # Rotate by a random angle between -10 and +10 degrees.
    transforms.ToTensor(),                # Pixels become floating-point values in [0, 1].
    transforms.Normalize(MEAN, STD),       # Match the pretrained weights' RGB scaling.
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # Resize to 150 x 150.
    transforms.ToTensor(),                       # Pixels become values in [0, 1].
    transforms.Normalize(MEAN, STD),             # Same ImageNet scaling, no random changes.
])

# ImageFolder assigns integer labels using alphabetically sorted folder names.
train_view = datasets.ImageFolder(flower_root, transform=train_transform)  # All flowers, training transform.
eval_view = datasets.ImageFolder(flower_root, transform=eval_transform)    # All flowers, evaluation transform.
class_names = train_view.classes                 # ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips'].
num_classes = len(class_names)                   # 5 flower classes.
assert train_view.class_to_idx == eval_view.class_to_idx  # Both views use the same labels.
print('Class mapping:', train_view.class_to_idx) # daisy=0, dandelion=1, roses=2, sunflowers=3, tulips=4

## 4. Separate training, validation, and test data

We use the same stratified split as Problem 2 (same seed and proportions):

- **Training:** 2,569 images (70%) used to update weights.
- **Validation:** 550 images (15%) used to select the best weights and decide when to stop.
- **Test:** 551 images (15%) reserved for the final evaluation, not for choosing settings.

The stratified split keeps the class proportions balanced. `Subset` selects images by index. `DataLoader` supplies batches; only the training batches are shuffled.

In [ ]:
all_indices = list(range(len(eval_view)))        # One index per photo.
train_indices, remaining = train_test_split(     # First split: 70% training, 30% set aside,
    all_indices, test_size=0.30,
    random_state=SEED, stratify=eval_view.targets,  # keeping class proportions equal.
)
val_indices, test_indices = train_test_split(    # Second split: the 30% becomes 15% validation + 15% test.
    remaining, test_size=0.50,
    random_state=SEED, stratify=[eval_view.targets[i] for i in remaining],
)

train_data = Subset(train_view, train_indices)   # Training images WITH augmentation.
val_data = Subset(eval_view, val_indices)        # Validation images, no augmentation.
test_data = Subset(eval_view, test_indices)      # Test images, no augmentation.
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)  # Shuffled training batches.
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, num_workers=0)    # Validation batches, fixed order.
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, num_workers=0)  # Test batches, fixed order; ALL test images.

print('Training:', len(train_data), '| Validation:', len(val_data), '| Test:', len(test_data))  # Split sizes.
images, labels = next(iter(train_loader))        # One batch to check the shapes.
print('One batch:', images.shape, '| Labels:', labels.shape)  # Expect [20, 3, 150, 150] and [20].

## 5. Reuse VGG16 features and add our classifier

VGG16 has two main parts: convolutional layers that extract features, and a classifier that combines them into predictions. We keep the pretrained `.features` and replace the original classifier with a small **flower** classifier.

Our model: VGG16 features → Flatten → 256 neurons → ReLU → Dropout → **5 outputs** (one per flower class).

- `requires_grad = False` freezes the pretrained weights: they are used but not updated.
- A dummy image finds the feature-output size. At 150 × 150 input size, it is 512 × 4 × 4, or 8,192 numbers after flattening.
- The new Linear layers start with randomly initialized weights.
- ReLU adds nonlinearity. Dropout temporarily disables some activations during training to help reduce overfitting.
- **Change for flowers:** the final layer outputs five raw scores (logits), one per class. The predicted class is the one with the highest score.
- `CrossEntropyLoss` replaces `BCEWithLogitsLoss` because there are five classes instead of two. It applies softmax internally, so do not add softmax before this loss; use softmax later when displaying probabilities.

In [ ]:
pretrained = models.vgg16(weights=models.VGG16_Weights.DEFAULT)  # VGG16 with ImageNet weights.
conv_base = pretrained.features                  # Keep only the convolutional part.
del pretrained  # Release the large original classifier that we do not need.

# Freeze all pretrained convolution weights for stage 1.
for parameter in conv_base.parameters():
    parameter.requires_grad = False              # Used, but not updated.

with torch.no_grad():                            # No gradients needed for the shape check.
    feature_shape = conv_base(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)).shape  # Pass one dummy image.
    # Count the values from one image after flattening.
    number_of_features = feature_shape[1] * feature_shape[2] * feature_shape[3]

model = nn.Sequential(
    conv_base,                                   # Pretrained VGG16 feature extractor.
    nn.Flatten(),                                # 512 x 4 x 4 -> 8,192 numbers.
    nn.Linear(number_of_features, 256),          # 8,192 -> 256 hidden units.
    nn.ReLU(),                                   # Nonlinearity.
    nn.Dropout(0.5),                             # Randomly drop half the activations during training.
    nn.Linear(256, num_classes),  # Five logits, one per flower class; no softmax before CrossEntropyLoss.
).to(device)                                     # Move the model to the GPU/CPU.
loss_function = nn.CrossEntropyLoss()            # Multi-class loss (replaces BCEWithLogitsLoss).
print('Convolution output:', tuple(feature_shape))  # (1, 512, 4, 4)
print('Flattened features:', number_of_features)    # 8192
print('Total parameters:', sum(p.numel() for p in model.parameters()))                          # All weights.
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))   # Only the new classifier.

## 6. Define what happens during one pass through the data

This is the only batch-processing loop in the program. We reuse it for training, validation, and testing.

If an optimizer is supplied, we turn on training behavior, calculate gradients, and update weights. Otherwise we use evaluation mode without gradients. Dropout is active only during training. This VGG16 variant has no BatchNorm layers.

**Changes for flowers:** the model returns five logits per image, so there is no `squeeze(1)`. `CrossEntropyLoss` needs integer (`long`) labels. The predicted class is `logits.argmax(1)`, the class with the highest score.

In [ ]:
def run_epoch(loader, optimizer=None):           # One pass through a loader; trains only if an optimizer is given.
    training = optimizer is not None             # True for training, False for validation/testing.
    model.train(training)  # True enables dropout; False is equivalent to model.eval().
    total_loss, correct, count = 0.0, 0, 0       # Running totals.
    # Gradients are needed for training but not for validation or testing.
    with torch.set_grad_enabled(training):
        for images, labels in loader:            # For each batch:
            images = images.to(device)           # Move images to the device.
            labels = labels.to(device)           # Integer class labels 0..4 for CrossEntropyLoss.
            if training:
                optimizer.zero_grad()                  # Clear previous gradients.
            logits = model(images)                      # Five scores per image.
            loss = loss_function(logits, labels)        # Compare scores with labels.
            if training:
                loss.backward()                        # Compute gradients.
                optimizer.step()                       # Update trainable weights.
            total_loss += loss.item() * len(labels)     # Handle a smaller final batch correctly.
            correct += (logits.argmax(1) == labels).sum().item()  # Highest score = predicted class.
            count += len(labels)                        # Count the images.
    return total_loss / count, correct / count   # Mean loss and accuracy.

## 7. Repeat training and keep the best weights

`fit()` calls `run_epoch()` once for training and once for validation each epoch. It records losses and accuracies so we can plot them later.

Each call creates a new optimizer using only trainable parameters. This is important when stage 2 unfreezes additional layers.

We save a copy of the weights whenever validation loss improves. Early stopping ends a stage after three consecutive epochs without improvement. At the end, we restore the best weights, not necessarily the last epoch's weights.

The starting model is also considered. This protects the stage-1 model if fine-tuning does not improve validation loss. "Epoch 0" means the starting weights remained best.

In [ ]:
def fit(epochs, learning_rate, stage_name):      # Train for up to `epochs`; return the history.
    # Create a new optimizer after changing which parameters are frozen.
    optimizer = torch.optim.RMSprop(
        [p for p in model.parameters() if p.requires_grad], lr=learning_rate,  # Only trainable weights.
        alpha=0.9, eps=1e-7,  # Control gradient averaging and numerical stability.
    )
    history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}  # Metrics per epoch.

    # Evaluate the incoming model before changing any weights.
    best_loss, _ = run_epoch(val_loader)         # Starting validation loss.
    best_weights = copy.deepcopy(model.state_dict())  # Starting weights.
    best_epoch, waiting = 0, 0                   # Best epoch so far; epochs without improvement.

    for epoch in range(1, epochs + 1):           # Repeat for every epoch:
        train_loss, train_accuracy = run_epoch(train_loader, optimizer)  # Learn from the training images.
        val_loss, val_accuracy = run_epoch(val_loader)                   # Check on the validation images.
        history['train_loss'].append(train_loss)          # Record the four metrics.
        history['val_loss'].append(val_loss)
        history['train_accuracy'].append(train_accuracy)
        history['val_accuracy'].append(val_accuracy)

        # Select weights using validation loss, never test results.
        if val_loss < best_loss:                 # Improvement:
            best_loss = val_loss                 # remember the new best loss,
            best_weights = copy.deepcopy(model.state_dict())  # copy the weights,
            best_epoch, waiting = epoch, 0       # and reset the waiting counter.
        else:
            waiting += 1                         # No improvement this epoch.

        print(f'{stage_name} {epoch}/{epochs} | Train: {train_accuracy:.2%}'
              f' | Validation: {val_accuracy:.2%} | Val loss: {val_loss:.4f}', flush=True)  # Progress line.
        if waiting >= PATIENCE:                  # Too many epochs without improvement:
            print('Early stopping: validation loss stopped improving.')
            break                                # stop this stage.

    # Undo any later epochs that performed worse on validation data.
    model.load_state_dict(best_weights)          # Restore the best weights.
    print(f'{stage_name}: restored best weights from epoch {best_epoch}.')
    return history                               # Losses and accuracies per epoch.

In [ ]:
# How does our Model look
print(model)                                     # VGG16 features followed by the new 5-class classifier.

## 8. Stage 1 — train only the new classifier

The convolutional base is frozen. Only the new Linear layers learn, while Dropout discourages reliance on individual hidden features. The validation result printed after fit belongs to the restored best model, not necessarily the final epoch shown in the history.

**Added for Problem 3:** we keep a copy of the stage-1 weights (`stage1_weights`) so that the stage-1 model can be evaluated on the test set in section 11, after stage 2 has changed the model.

In [ ]:
history_initial = fit(INITIAL_EPOCHS, INITIAL_LR, 'Frozen base')  # Stage 1 training.
initial_val_loss, initial_val_accuracy = run_epoch(val_loader)    # Validation result of the selected stage-1 model.
print(f'Selected stage-1 validation accuracy: {initial_val_accuracy:.2%}')
stage1_weights = copy.deepcopy(model.state_dict())  # Keep the stage-1 model for the test evaluation in section 11.

## 9. Stage 2 — fine-tune the final convolutional block

VGG16 has five convolutional blocks. `conv_base[24:]` selects block 5, starting at module index 24. It contains three convolutions, their ReLUs, and pooling.

We make this block trainable while keeping blocks 1–4 frozen. The classifier also remains trainable. We reuse the weights learned in stage 1; we do not create a new model.

Calling `fit()` again creates an optimizer that includes the newly unfrozen parameters. The smaller learning rate helps adjust pretrained features without changing them too abruptly.

In [ ]:
# Module index 24 starts block 5 (it is not the 24th convolution).
for parameter in conv_base[24:].parameters():
    parameter.requires_grad = True               # Unfreeze block 5.

print('Trainable convolution modules:')
for index, layer in enumerate(conv_base):        # List the convolutions that will now learn.
    if isinstance(layer, nn.Conv2d) and layer.weight.requires_grad:
        print(f'  features[{index}]: {layer}')
print('Total trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

history_finetune = fit(FINETUNE_EPOCHS, FINETUNE_LR, 'Fine-tuning')  # Stage 2 training.
final_val_loss, final_val_accuracy = run_epoch(val_loader)           # Validation result of the selected stage-2 model.
print(f'Selected final validation accuracy: {final_val_accuracy:.2%}')
stage2_weights = copy.deepcopy(model.state_dict())  # Keep the stage-2 model (used again in section 11).

## 10. Plot both stages

Training metrics are measured while weights change, with augmentation and dropout active. Validation metrics use fixed weights without either effect, so their difference is not a pure measure of overfitting.

The dashed line marks the transition to fine-tuning. Stage 2 starts from the best stage-1 weights, which may differ from its last plotted epoch. These are learning curves, not test results.

In [ ]:
# Join the two histories to show the complete learning process.
history = {key: history_initial[key] + history_finetune[key] for key in history_initial}
epoch_numbers = range(1, len(history['train_loss']) + 1)   # x-axis: 1, 2, ...
boundary = len(history_initial['train_loss']) + 0.5        # Where stage 2 begins.

fig, axes = plt.subplots(1, 2, figsize=(11, 4))            # Loss and accuracy side by side.
for ax, metric in zip(axes, ['loss', 'accuracy']):
    ax.plot(epoch_numbers, history['train_' + metric], label='Training')    # Training curve.
    ax.plot(epoch_numbers, history['val_' + metric], label='Validation')    # Validation curve.
    ax.axvline(boundary, linestyle='--', color='gray', label='Fine-tuning begins')  # Stage boundary.
    ax.set(xlabel='Epoch', ylabel=metric.capitalize(), title=metric.capitalize())   # Labels and title.
    ax.legend()                                            # Legend.
axes[1].set_ylim(0, 1)                                     # Accuracy axis from 0 to 1.
plt.tight_layout()                                         # Reduce overlap.
plt.savefig(output_dir / 'learning_curves.png', dpi=150)   # Save the figure.
plt.show()                                                 # Display it.

## 11. Evaluate the stage 1 and stage 2 models on the reserved test set

The test images are now used for the first time. We also measure training accuracy without augmentation or dropout so the reported training result is clearly defined.

**Problem 3:** we evaluate **both** models on the **entire test set** (all 551 images): first the stage-1 weights saved in section 8, then the final stage-2 weights. For each, `test_report()` prints the classification report and confusion matrix and computes **overall precision**:

- **Macro precision:** the average of the five per-class precisions (each class counts equally).
- **Weighted precision:** each class weighted by its number of test images.

Precision asks how often a predicted class is correct. Recall asks how many actual members of a class were found. F1 combines precision and recall; support is the number of images. The confusion matrix has actual classes in rows and predictions in columns. Do not tune settings using this test report.

In [ ]:
def test_report(stage_name):                     # Evaluate the current model on the ENTIRE test set.
    test_loss, test_accuracy = run_epoch(test_loader)  # Test loss and accuracy.
    actual_labels, predictions = [], []          # Collected over all test images.
    model.eval()                                 # Evaluation mode (no dropout).
    with torch.no_grad():                        # No gradients needed.
        for images, labels in test_loader:       # Every test batch, so every test image:
            logits = model(images.to(device))    # five scores per image,
            predictions.extend(logits.argmax(1).cpu().tolist())  # predicted class = highest score,
            actual_labels.extend(labels.tolist())                # true class.
    assert len(actual_labels) == len(test_data)  # Confirm the entire test set was evaluated.
    precision_macro = precision_score(actual_labels, predictions, average='macro', zero_division=0)        # Classes count equally.
    precision_weighted = precision_score(actual_labels, predictions, average='weighted', zero_division=0)  # Weighted by class size.
    print(f'===== {stage_name} | test images evaluated: {len(actual_labels)} of {len(test_data)} =====')
    print(f'Test accuracy: {test_accuracy:.2%} | Test loss: {test_loss:.4f}')
    print(f'Overall precision (macro): {precision_macro:.2%} | Overall precision (weighted): {precision_weighted:.2%}')
    print(classification_report(actual_labels, predictions, labels=list(range(num_classes)),
                                target_names=class_names, zero_division=0))  # Per-class precision, recall, F1, support.
    ConfusionMatrixDisplay.from_predictions(actual_labels, predictions, labels=list(range(num_classes)),
                                            display_labels=class_names, cmap='Blues', colorbar=False)  # Rows = actual, columns = predicted.
    plt.title(f'Test confusion matrix – {stage_name}')
    plt.tight_layout()
    plt.show()
    return {'test_accuracy': test_accuracy, 'test_loss': test_loss,
            'precision_macro': precision_macro, 'precision_weighted': precision_weighted}  # Results for the comparison.

# A separate view of training files removes random augmentation for final evaluation.
train_eval_data = Subset(eval_view, train_indices)  # Training images with the evaluation transform.
train_eval_loader = DataLoader(train_eval_data, batch_size=BATCH_SIZE)
_, final_train_accuracy = run_epoch(train_eval_loader)  # Final (stage-2) model's training accuracy.
print(f'Final train accuracy (evaluation mode): {final_train_accuracy:.2%}')
print(f'Validation accuracy: {final_val_accuracy:.2%}')

model.load_state_dict(stage1_weights)            # Switch to the stage-1 model.
stage1_results = test_report('Stage 1 (frozen base)')
model.load_state_dict(stage2_weights)            # Switch back to the final stage-2 model.
stage2_results = test_report('Stage 2 (fine-tuned)')

print(f"{'Model':<24}{'Precision (macro)':>19}{'Precision (weighted)':>22}{'Accuracy':>10}")  # Comparison table header.
for stage_name, results in [('Stage 1 (frozen base)', stage1_results), ('Stage 2 (fine-tuned)', stage2_results)]:
    print(f"{stage_name:<24}{results['precision_macro']:>19.2%}{results['precision_weighted']:>22.2%}"
          f"{results['test_accuracy']:>10.2%}")  # One row per stage.

## 12. Predict one image of every flower class and save the model

**Problem 3:** we test the final (stage 2) model with **one test image of each of the five flower classes**: the first test image of each class.

Apply the same validation/test preprocessing to each photo. `unsqueeze(0)` adds the batch dimension because the model expects a batch, even for one image.

Softmax converts the five output logits to class probabilities that add up to 1. These are model estimates, not guarantees. The title is green when the prediction is correct and red when it is wrong.

Save the learned parameters in a `state_dict`, along with the class names and preprocessing settings. To reload it later, first recreate this same model architecture, then load its saved weights.

In [ ]:
# Use the first test image of each class to demonstrate prediction on single photos.
fig, axes = plt.subplots(1, num_classes, figsize=(4 * num_classes, 4))  # One panel per flower class.
model.eval()                                     # Evaluation mode (no dropout).
for axis, class_index in zip(axes, range(num_classes)):  # For each class:
    photo_index = next(i for i in test_indices if eval_view.targets[i] == class_index)  # first test image of this class,
    photo_path, actual_label = eval_view.samples[photo_index]  # its file path and true label.
    with Image.open(photo_path) as photo:        # Open the photo file.
        photo = photo.convert('RGB')             # Make sure it has three color channels.
        input_image = eval_transform(photo).unsqueeze(0).to(device)  # Preprocess and add the batch dimension.
        with torch.no_grad():
            probabilities = torch.softmax(model(input_image), dim=1)[0]  # Five class probabilities.
        predicted_label = probabilities.argmax().item()  # Most likely class.
        axis.imshow(photo)                       # Show the photo.
    axis.set_title(f'Actual: {class_names[actual_label]}\nPredicted: {class_names[predicted_label]}'
                   f' ({probabilities[predicted_label]:.1%})',
                   color='green' if predicted_label == actual_label else 'red')  # Green = correct, red = wrong.
    axis.axis('off')                             # Hide axis ticks.
    print(f'{class_names[actual_label]:<11} -> predicted {class_names[predicted_label]:<11} | probabilities: '
          + ', '.join(f'{name} {p:.1%}' for name, p in zip(class_names, probabilities.tolist())))  # All five probabilities.
fig.suptitle('Final model: one test image of every flower class')
plt.tight_layout()
plt.show()

# Save CPU tensors so the checkpoint can be loaded on other devices.
torch.save({
    'state_dict': {name: tensor.cpu() for name, tensor in model.state_dict().items()},  # Final weights.
    'class_names': class_names, 'image_size': IMAGE_SIZE,
    'mean': MEAN, 'std': STD, 'flattened_features': number_of_features,
}, output_dir / 'vgg16_finetuned.pth')
(output_dir / 'metrics.json').write_text(json.dumps({  # Human-readable metrics.
    'initial_val_accuracy': initial_val_accuracy,
    'final_train_accuracy': final_train_accuracy,
    'final_val_accuracy': final_val_accuracy,
    'stage1_test': stage1_results, 'stage2_test': stage2_results,  # Problem 3: test precision of both stages.
    'history_initial': history_initial, 'history_finetune': history_finetune,
}, indent=2))
print('Saved model, metrics, and plot to:', output_dir.resolve())

## Takeaways: transfer learning in two stages

| Stage | Frozen weights | Weights that learn |
|---|---|---|
| Train the new classifier | All VGG16 convolutional blocks | New classifier |
| Fine-tune | VGG16 blocks 1–4 | Block 5 and the classifier |

1. Start with features learned from a large image dataset.
2. Prepare inputs with the normalization expected by the pretrained model.
3. Train the new classifier before unfreezing pretrained layers.
4. Use a smaller learning rate for fine-tuning.
5. Use validation results to select weights; reserve test results for final evaluation.

Fine-tuning may help, but improvement is not guaranteed. This notebook keeps the model with the lowest validation loss.

## 13. Report on results

_Fill in this section after running the notebook, using the outputs of sections 11 and 12._

**Precision on the full test set (551 images)**

| Model | Precision (macro) | Precision (weighted) | Test accuracy |
|:---|:---:|:---:|:---:|
| Stage 1 (frozen VGG16 base) | | | |
| Stage 2 (fine-tuned block 5) | | | |

Points to discuss:

1. Did fine-tuning (stage 2) improve precision over stage 1, and by how much?
2. Which flower classes have the highest and lowest precision? Which classes are confused with each other (confusion matrix)?
3. How did the five single-image tests in section 12 go?
4. How does VGG16 transfer learning compare with the small CNN from Problem 2 (best: 75.8% macro precision)?